In [1]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain.agents import create_agent


# 🔐 Load API keys
load_dotenv(".env")

google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

print("Google API key loaded:", bool(google_api_key))


# 🔸 Initialize Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=google_api_key
)


# ✅ Tool 1: Simple QA Tool
def simple_qa(question):
    response = llm.invoke(question)
    return response.content


qa_tool = Tool(
    name="simple_qa",
    func=simple_qa,
    description="Answers factual questions clearly"
)


# ✅ Tool 2: Summarizer Tool
def summarizer(text):
    prompt = f"Summarize this text:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content


summary_tool = Tool(
    name="summarizer",
    func=summarizer,
    description="Summarizes long paragraphs or text content"
)


# ✅ Tool 3: Web Search Tool
search_tool = TavilySearchResults(
    max_results=3
)


# 🔧 Put all tools together
tools = [
    qa_tool,
    summary_tool,
    search_tool
]


# 🤖 Create Agent
agent = create_agent(
    model=llm,
    tools=tools
)


# 🚀 Run user queries
queries = [
    "What is LangGraph in LangChain?"
    # "Summarize this: LangChain is a framework to build LLM apps using prompts, memory, tools, and agents.",
    # "Latest news about OpenAI GPT-4o"
]


for query in queries:

    print("\n🧑‍💻 User Query:", query)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    print(
        "\n🤖 Agent Response:",
        response["messages"][-1].content
    )

C:\Users\khavy\AppData\Local\Temp\ipykernel_20716\2159786294.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import TavilySearchResults


Google API key loaded: True


C:\Users\khavy\AppData\Local\Temp\ipykernel_20716\2159786294.py:56: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(



🧑‍💻 User Query: What is LangGraph in LangChain?


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



🤖 Agent Response: [{'type': 'text', 'text': 'LangGraph is an extension of the LangChain framework specifically designed for building stateful, multi-actor applications with LLMs, particularly those that involve cycles (loops) and complex decision-making.\n\nWhile LangChain provides the fundamental building blocks like LLMs, prompts, chains, tools, and agents, LangGraph offers a robust framework for orchestrating complex, dynamic workflows that go beyond simple linear chains. It acts like a finite state machine or a directed acyclic graph (DAG), but with the crucial ability to handle loops and explicit state management.\n\nLangGraph was created to address challenges in traditional LangChain agents, such as:\n\n*   **Statefulness:** It provides a clear way to define and manage the "state" of your application, allowing agents to remember past interactions and decisions.\n*   **Cycles/Loops:** It explicitly supports defining cyclical workflows, where an agent might repeat steps based on o

In [2]:
# An AI agent uses an LLM (Gemini) as its brain and tools as its abilities. The agent decides which tool to use based on the user's request.

                 User
                  ↓
          "What should I do?"
                  ↓
              AI Agent
                  ↓
            Gemini LLM
               (Brain)
                  ↓
       ┌──────────┼──────────┐
       ↓          ↓          ↓
   simple_qa  summarizer  web_search
       ↓          ↓          ↓
   Answer      Summary     Internet
       └──────────┼──────────┘
                  ↓
             Final Answer

